# 03 · Prophet en el top 10: revisión visual del modelado (5.5 y 5.6)

**Objetivo.** Revisar visualmente los resultados de Prophet frente al modelo base, en los 10 productos del notebook `02_stl_top10`. El notebook **no entrena ni recalcula** modelos: lee lo que ya produjeron `python -m src.models.baseline` y `python -m src.models.prophet_model` en `data/resultados/` y, de Supabase, solo la serie de entrenamiento para graficarla.

**Cómo ejecutarlo.**
1. Desde la raíz del repositorio, con el entorno virtual activo y el archivo `.env` con `DATABASE_URL`.
2. Haber corrido antes los dos módulos de `src/models/` (generan los CSV y el JSON de `data/resultados/`).
3. Ejecutar las celdas en orden.

**Confidencialidad.** Las salidas muestran ventas por producto: **limpiar las salidas antes de cualquier commit**.

**Contenido.** 1 Configuración · 2 Serie de entrenamiento y atípicos · 3 Parámetros elegidos · 4 Pronósticos de 2025 · 5 Métricas por producto · 6 WAPE frente al umbral · 7 Resumen.

## 1. Configuración

Periodos, horizontes y productos salen de `src/config.py`, el mismo módulo que usan el modelo base y Prophet; aquí no se repite ningún valor. El umbral de WAPE se calcula con el WAPE global del base guardado en `metricas_base.csv`.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Raíz del proyecto (para importar src/ aunque el notebook se abra desde notebooks/)
RAIZ = Path.cwd().resolve()
while not (RAIZ / "src").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

from src import config
from src.models import evaluation as ev

RESULTADOS = config.RESULTS_DIR
CLAVES = list(config.TOP10_CLAVES)

# Colores (paleta validada para lectores con daltonismo)
TINTA, GRIS, LINEA_GUIA = "#0b0b0b", "#898781", "#c3c2b7"
AZUL, NARANJA = "#2a78d6", "#eb6834"
AZUL_H = {1: "#104281", 2: "#2a78d6", 3: "#86b6ef"}   # h = 1, 2, 3: de oscuro a claro
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": LINEA_GUIA, "axes.labelcolor": "#52514e",
                     "xtick.color": GRIS, "ytick.color": GRIS, "font.size": 9})

metricas_base = pd.read_csv(RESULTADOS / "metricas_base.csv", dtype={"clave": str, "horizonte": str})
umbral = ev.wape_threshold(metricas_base)

display(pd.Series({
    "train": f"{config.TRAIN_START:%Y-%m} a {config.TRAIN_END:%Y-%m}",
    "validación interna (elegir parámetros)": f"{config.VALIDATION_START:%Y-%m} a {config.VALIDATION_END:%Y-%m}",
    "test": f"{config.TEST_START:%Y-%m} a {config.TEST_END:%Y-%m}",
    "horizontes (meses)": ", ".join(map(str, config.HORIZONS)),
    "productos": ", ".join(CLAVES),
    "umbral de WAPE (%)": f"{umbral:.2f}",
}, name="valor").to_frame())

## 2. Serie de entrenamiento y atípicos tratados

Ventas mensuales en piezas de 2022 a 2024, leídas de Supabase (solo lectura). El límite de Tukey (Q3 + 3 × RIC sobre los meses con venta de train) se calcula con las mismas funciones que usan los modelos.

En cada panel, la línea punteada gris es el límite del producto. En los 4 meses atípicos, el punto **naranja** es el valor original y el punto **azul** el valor tratado (el menor entre el original y el límite). Los modelos candidatos, como Prophet, se **ajustan** con el valor tratado, pero su error se **mide** contra el original. El **modelo base** no usa el valor tratado: promedia los valores originales, porque replica el método actual de la empresa, que no trata atípicos.

In [ ]:
from src.db.repository import Repositorio, conectar

conn = conectar()
try:
    repo = Repositorio(conn)
    unidad = repo.productos().astype({"clave": str}).set_index("clave")["unidad"]
    ventas_train = repo.ventas(config.TRAIN_START, config.TRAIN_END)
finally:
    conn.close()                       # solo se leía

assert (unidad[CLAVES] == "PZA").all(), "Algún producto del top 10 no está en piezas"
train = ev.build_monthly_matrix(ventas_train, CLAVES, config.TRAIN_START, config.TRAIN_END)
limites = ev.train_limits(train)
tratada = ev.treat_outliers(train, config.OUTLIER_MONTHS, limites)

display(ev.outlier_report(train, tratada, config.OUTLIER_MONTHS, limites)
        .style.format({"original": "{:,.0f}", "limite": "{:,.1f}", "tratado": "{:,.1f}"}).hide(axis="index"))
otros = ev.other_exceedances(train, config.OUTLIER_MONTHS, limites)
print("Otros meses de train sobre su límite (no se tratan):", "ninguno" if otros.empty else "")
if not otros.empty:
    display(otros)

In [ ]:
fig, ejes = plt.subplots(5, 2, figsize=(10, 12), sharex=True)
for ax, c in zip(ejes.flat, CLAVES):
    ax.plot(train.index, train[c], color=AZUL, lw=1.6)
    ax.axhline(limites[c], color=GRIS, lw=1, ls=":")
    for clave, mes in config.OUTLIER_MONTHS:
        if clave != c:
            continue
        f = pd.Timestamp(mes)
        ax.vlines(f, tratada.loc[f, c], train.loc[f, c], color=GRIS, lw=1)
        ax.scatter(f, train.loc[f, c], s=40, color=NARANJA, zorder=3, edgecolor="white", linewidth=1.5)
        ax.scatter(f, tratada.loc[f, c], s=40, color=AZUL, zorder=3, edgecolor="white", linewidth=1.5)
    ax.set_title(f"Producto {c}", loc="left", fontsize=10, color=TINTA)
    ax.grid(axis="y", color="#e1e0d9", lw=0.6)
    ax.set_ylim(bottom=0)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
for ax in ejes[:, 0]:
    ax.set_ylabel("piezas")
leyenda = [plt.Line2D([], [], color=AZUL, lw=1.6, label="ventas (train)"),
           plt.Line2D([], [], color=GRIS, lw=1, ls=":", label="límite de Tukey"),
           plt.Line2D([], [], marker="o", ls="", color=NARANJA, label="atípico: valor original"),
           plt.Line2D([], [], marker="o", ls="", color=AZUL, label="atípico: valor tratado")]
fig.legend(handles=leyenda, loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 1.0))
fig.suptitle("Serie de entrenamiento (2022–2024) con los atípicos tratados", x=0.01, ha="left", y=1.03,
             fontsize=11, color=TINTA)
fig.tight_layout()
plt.show()

## 3. Parámetros elegidos de Prophet

Para cada producto se probaron 15 combinaciones (estacionalidad ninguna, Fourier o indicadoras del mes; modo aditivo o multiplicativo; `changepoint_prior_scale` 0.01, 0.05 o 0.2) con validación interna en 2024: origen móvil, h = 1, 2 y 3, 36 pronósticos. Se eligió la de menor MAE.

La **referencia** es la combinación más simple (sin estacionalidad, `changepoint_prior_scale = 0.01`). La última columna indica cuánto mejora la elegida frente a ella en la validación; un valor cercano a 0 significa que la estacionalidad casi no ayudó en ese producto.

In [ ]:
parametros = json.loads((RESULTADOS / "prophet_parametros.json").read_text(encoding="utf-8"))
print("Prophet", parametros["meta"]["prophet"], "· semilla", parametros["meta"]["semilla"],
      "· validación", parametros["meta"]["validacion"])
tabla_param = pd.DataFrame.from_dict(parametros["productos"], orient="index").loc[CLAVES]
tabla_param["mejora vs referencia"] = 1 - tabla_param["mae_validacion"] / tabla_param["mae_referencia"]
display(tabla_param.rename_axis("clave").style.format({
    "changepoint_prior_scale": "{:g}", "mae_validacion": "{:.2f}", "mae_referencia": "{:.2f}",
    "mejora vs referencia": "{:.1%}"}, na_rep="—"))
print(tabla_param["estacionalidad"].value_counts().to_string())

## 4. Pronósticos de 2025 por producto

Cada panel muestra la venta real de 2025 (negro), el modelo base (gris punteado; es el mismo en los tres horizontes y usa los valores originales, con atípicos) y Prophet con 1, 2 y 3 meses de anticipación (azul, de oscuro a claro). Un pronóstico a 3 meses de marzo se hizo con datos hasta diciembre; a 1 mes, con datos hasta febrero.

In [ ]:
cols = {"dtype": {"clave": str}, "parse_dates": ["mes"]}
base = pd.read_csv(RESULTADOS / "base_top10.csv", **cols)
prophet = pd.read_csv(RESULTADOS / "prophet_top10.csv", **cols)
assert len(base) == len(prophet) == len(CLAVES) * len(config.HORIZONS) * 12

fig, ejes = plt.subplots(5, 2, figsize=(10, 13), sharex=True)
for ax, c in zip(ejes.flat, CLAVES):
    b = base[(base["clave"] == c) & (base["horizonte"] == 1)]
    ax.plot(b["mes"], b["real"], color=TINTA, lw=2, marker="o", ms=4)
    ax.plot(b["mes"], b["pronostico"], color=GRIS, lw=1.6, ls="--")
    for h in config.HORIZONS:
        p = prophet[(prophet["clave"] == c) & (prophet["horizonte"] == h)]
        ax.plot(p["mes"], p["pronostico"], color=AZUL_H[h], lw=1.6)
    ax.set_title(f"Producto {c}", loc="left", fontsize=10, color=TINTA)
    ax.grid(axis="y", color="#e1e0d9", lw=0.6)
    ax.set_ylim(bottom=0)
for ax in ejes[:, 0]:
    ax.set_ylabel("piezas")
for ax in ejes[-1]:
    ax.tick_params(axis="x", rotation=45)
leyenda = [plt.Line2D([], [], color=TINTA, lw=2, marker="o", ms=4, label="real"),
           plt.Line2D([], [], color=GRIS, lw=1.6, ls="--", label="base")]
leyenda += [plt.Line2D([], [], color=AZUL_H[h], lw=1.6, label=f"Prophet h={h}") for h in config.HORIZONS]
fig.legend(handles=leyenda, loc="upper center", ncol=5, frameon=False, bbox_to_anchor=(0.5, 1.0))
fig.suptitle("2025: venta real frente al base y a Prophet (piezas)", x=0.01, ha="left", y=1.03,
             fontsize=11, color=TINTA)
fig.tight_layout()
plt.show()

## 5. Métricas por producto: Prophet frente al base

Cada fila resume los 36 pronósticos de 2025 de un producto (h = 1, 2 y 3 juntos). La **reducción** es `1 − Prophet / base`: positiva significa que Prophet se equivoca menos que el base y negativa que se equivoca más. El criterio de éxito pide una reducción de al menos 15 % y un MAPE de 20 % o menos. Qué métrica cuenta para el 15 % está pendiente de definir con el equipo, así que se muestran todas.

In [ ]:
metricas_prophet = pd.read_csv(RESULTADOS / "metricas_prophet.csv", dtype={"clave": str, "horizonte": str})
comp = ev.comparison_table(metricas_prophet, metricas_base)
por_producto = comp[comp["nivel"] == "producto"].set_index("clave").loc[CLAVES]
columnas = ["mae", "mae_base", "red_mae", "rmse", "rmse_base", "red_rmse", "mape", "mape_base", "red_mape",
            "wape", "wape_base", "red_wape", "cumple_mape"]
display(por_producto[columnas].style.format(
    {c: "{:.2f}" for c in columnas if not c.startswith("red_") and c != "cumple_mape"}
    | {c: "{:+.1%}" for c in columnas if c.startswith("red_")}))
print(f"Cumplen MAPE ≤ {config.MAPE_TARGET:g} %: {', '.join(por_producto.index[por_producto['cumple_mape']])}")

por_h = comp[comp["nivel"] == "producto_horizonte"]
print("\nMAE de Prophet por producto y horizonte:")
display(por_h.pivot(index="clave", columns="horizonte", values="mae").loc[CLAVES].style.format("{:.2f}"))

## 6. WAPE global y por horizonte frente al umbral

El WAPE suma los errores absolutos de los 10 productos y los divide entre la venta real total (360 pronósticos en el global y 120 en cada horizonte). La línea punteada es el umbral fijado con el base **antes** de entrenar Prophet: el WAPE del base multiplicado por 0.85. Una barra de Prophet por debajo de la línea cumple el criterio.

In [ ]:
g = comp[comp["nivel"].isin(["global", "global_horizonte"])].copy()
g["etiqueta"] = ["global" if h == ev.ALL else f"h = {h}" for h in g["horizonte"]]

fig, ax = plt.subplots(figsize=(8, 4.2))
x = range(len(g))
ancho = 0.38
barras_b = ax.bar([i - ancho / 2 - 0.01 for i in x], g["wape_base"], ancho, color=GRIS, label="base")
barras_p = ax.bar([i + ancho / 2 + 0.01 for i in x], g["wape"], ancho, color=AZUL, label="Prophet")
for barras in (barras_b, barras_p):
    ax.bar_label(barras, fmt="%.3f", padding=2, fontsize=8, color="#52514e",
                  bbox={"facecolor": "white", "edgecolor": "none", "pad": 0.5})
ax.axhline(umbral, color=TINTA, lw=1, ls="--", label=f"umbral {umbral:.3f} %")
ax.set_xticks(list(x), g["etiqueta"])
ax.set_ylabel("WAPE (%)")
ax.set_ylim(0, g["wape_base"].max() * 1.15)
ax.grid(axis="y", color="#e1e0d9", lw=0.6)
ax.set_axisbelow(True)
ax.legend(frameon=False, loc="upper center", ncol=3, bbox_to_anchor=(0.5, -0.1))
ax.set_title("WAPE de 2025: base frente a Prophet", loc="left", fontsize=11, color=TINTA)
fig.tight_layout()
plt.show()

display(g.set_index("etiqueta")[["n", "wape_base", "wape", "red_wape", "cumple_umbral_wape"]]
                .style.format({"wape_base": "{:.4f}", "wape": "{:.4f}", "red_wape": "{:+.1%}"}))

## 7. Resumen: qué productos mejoran y cuáles empeoran

Se clasifica cada producto por la reducción de MAE frente al base (la misma clasificación sale con WAPE, porque el real es el mismo para ambos modelos). La última columna cuenta en cuántas de las cuatro métricas Prophet mejora al base: un producto puede mejorar en MAE y empeorar en MAPE si falla más en los meses de venta baja.

In [ ]:
reds = ["red_mae", "red_rmse", "red_mape", "red_wape"]
resumen = por_producto[reds + ["cumple_mape"]].assign(
    resultado=lambda d: d["red_mae"].map(lambda r: "mejora ≥ 15 %" if r >= config.MIN_REDUCTION
                                         else ("mejora < 15 %" if r > 0 else "empeora")),
    metricas_que_mejoran=lambda d: (d[reds] > 0).sum(axis=1).astype(str) + " de 4",
).sort_values("red_mae", ascending=False)
display(resumen.style.format({c: "{:+.1%}" for c in reds}))
for etiqueta, grupo in resumen.groupby("resultado", sort=False):
    print(f"{etiqueta}: {', '.join(grupo.index)}")